खालील नोटबुक GitHub Copilot Chat ने स्वयंचलितपणे तयार केले आहे आणि ते फक्त प्रारंभिक सेटअपसाठी आहे


# प्रॉम्प्ट अभियांत्रिकी परिचय
प्रॉम्प्ट अभियांत्रिकी ही नैसर्गिक भाषा प्रक्रिया कार्यांसाठी प्रॉम्प्ट्स डिझाइन आणि ऑप्टिमाइझ करण्याची प्रक्रिया आहे. यात योग्य प्रॉम्प्ट्सची निवड करणे, त्यांची पॅरामीटर्स ट्यून करणे आणि त्याच्या कार्यक्षमतेचे मूल्यांकन करणे यांचा समावेश होतो. प्रॉम्प्ट अभियांत्रिकी NLP मॉडेल्समध्ये उच्च अचूकता आणि कार्यक्षमता साध्य करण्यासाठी महत्वाची आहे. या विभागात, आपण OpenAI मॉडेल्सचा वापर करून प्रॉम्प्ट अभियांत्रिकीचे मूलतत्त्व पाहणार आहोत.


### सराव १: टोकनायझेशन
OpenAI कडून उपलब्ध असलेल्या त्वरित टोकनायझर tiktoken वापरून टोकनायझेशनचा अभ्यास करा
अधिक उदाहरणांसाठी [OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb?WT.mc_id=academic-105485-koreyst) पहा.


In [ ]:
# EXERCISE:
# 1. Run the exercise as is first
# 2. Change the text to any prompt input you want to use & re-run to see tokens

import tiktoken

# Define the prompt you want tokenized
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

# Set the model you want encoding for
encoding = tiktoken.encoding_for_model("gpt-4o")

# Encode the text - gives you the tokens in integer form
tokens = encoding.encode(text)
print(tokens);

# Decode the integers to see what the text versions look like
[encoding.decode_single_token_bytes(token) for token in tokens]


### व्यायाम 2: OpenAI API किल्ली सेटअपची पडताळणी करा

तुमचा OpenAI एंडपॉइंट योग्यरित्या सेटअप झाला आहे का ते सत्यापित करण्यासाठी खालील कोड चालवा. हा कोड फक्त एक सोपा बेसिक प्रॉम्प्ट वापरून पूर्णता तपासतो. इनपुट `oh say can you see` हा `by the dawn's early light..` च्या सारखाच पूर्ण होईल.


In [ ]:
# Uses the OpenAI client against the Azure OpenAI (Microsoft Foundry) v1 endpoint
# with the Responses API. See https://aka.ms/openai/start

import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']

def get_completion(prompt):
    response = client.responses.create(
        model=deployment,
        input=prompt,
        temperature=0, # this is the degree of randomness of the model's output
        max_output_tokens=1024,
        store=False,
    )
    return response.output_text

## ---------- Call the helper method

### 1. Set primary content or prompt text
text = f"""
oh say can you see
"""

### 2. Use that in the prompt template below
prompt = f"""
```{text}```
"""

## 3. Run the prompt
response = get_completion(prompt)
print(response)


### व्यायाम 3: बनावटे
जेव्हा आपण LLM ला अशा विषयाबद्दल प्रॉम्प्टसाठी पूर्णता परत करण्यास सांगता जे कदाचित अस्तित्वात नसेल, किंवा अशा विषयांबद्दल जे त्याला माहीत नसतील कारण ते त्याच्या पूर्व-प्रशिक्षित डेटासेटच्या बाहेर आहे (अधिक अलीकडील), तेव्हा काय होते हे तपासा. वेगळा प्रॉम्प्ट किंवा वेगळे मॉडेल वापरल्यास प्रतिसाद कसा बदलतो ते पाहा.


In [ ]:

## Set the text for simple prompt or primary content
## Prompt shows a template format with text in it - add cues, commands etc if needed
## Run the completion 
text = f"""
generate a lesson plan on the Martian War of 2076.
"""

prompt = f"""
```{text}```
"""

response = get_completion(prompt)
print(response)

### व्यायाम ४: सूचनांवर आधारित 
प्रायमरी कंटेंट सेट करण्यासाठी "text" व्हेरिएबल वापरा 
आणि त्या प्रायमरी कंटेंटशी संबंधित सूचनांसाठी "prompt" व्हेरिएबल वापरा.

येथे आम्ही मॉडेलला दुसऱ्या वर्गाच्या विद्यार्थ्यासाठी मजकूराचा सारांश सांगायला सांगतो


In [ ]:
# Test Example
# https://platform.openai.com/playground/p/default-summarize

## Example text
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

## Set the prompt
prompt = f"""
Summarize content you are provided with for a second-grade student.
```{text}```
"""

## Run the prompt
response = get_completion(prompt)
print(response)

### सराव 5: गुंतागुंतीचा प्रॉम्प्ट  
असा विनंती करा ज्यात प्रणाली, उपयोगकर्ता आणि सहाय्यक संदेश आहेत  
प्रणाली सहाय्यक संदर्भ सेट करते  
उपयोगकर्ता व सहाय्यक संदेश बहु-पल्ट संवादाचा संदर्भ देतात  

ध्यान द्या की प्रणाली संदर्भात सहाय्यक व्यक्तिमत्व "व्यंगात्मक" म्हणून सेट केले आहे.  
वेगळे व्यक्तिमत्व संदर्भ वापरून पहा. किंवा इनपुट/आउटपुट संदेशांची वेगळी मालिका वापरून पहा  


In [ ]:
response = client.responses.create(
    model=deployment,
    input=[
        {"role": "system", "content": "You are a sarcastic assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"},
        {"role": "assistant", "content": "Who do you think won? The Los Angeles Dodgers of course."},
        {"role": "user", "content": "Where was it played?"}
    ],
    store=False,
)
print(response.output_text)


### सराव: तुमच्या अंतर्ज्ञानाचा अभ्यास करा
वरील उदाहरणे तुम्हाला नवीन प्रॉम्प्ट तयार करण्यासाठी वापरता येणारे नमुने देतात (सोप्या, गुंतागुंतीच्या, सूचना इत्यादी) - आपण यापुढे काही इतर कल्पना जसे की उदाहरणे, संकेत आणि अधिक यांचा अभ्यास करण्यासाठी इतर सराव तयार करण्याचा प्रयत्न करा.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
